In [2]:
import requests
import pandas as pd
import numpy as np
import datetime

# ========================
# 1️⃣ NOAA API setup
# ========================
TOKEN = "qDMWPIYGHTKRKkhCWbKjwkUQxNMinWWd"  # Replace with your NOAA token
headers = {"token": TOKEN}

# Set date range: past 1 year
end_date = datetime.datetime.now().date()
start_date = end_date - pd.Timedelta(days=365)

params = {
    "datasetid": "GHCND",
    "locationid": "FIPS:45079",  # Columbia, SC (Richland County)
    "startdate": start_date.isoformat(),
    "enddate": end_date.isoformat(),
    "datatypeid": "PRCP",
    "limit": 1000,
    "units": "metric"
}

# ========================
# 2️⃣ Pull data from NOAA
# ========================
response = requests.get(
    "https://www.ncei.noaa.gov/cdo-web/api/v2/data",
    headers=headers,
    params=params,
    timeout=10
)

data = response.json().get("results", [])
df = pd.DataFrame(data)

if df.empty:
    print("No data returned for this date range")
else:
    # convert to datetime and mm
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = df["value"] / 10  # tenths of mm → mm

    # ========================
    # 3️⃣ Aggregate stations by day
    # ========================
    daily_rain = df.groupby("date")["value"].sum().reset_index()

    # ========================
    # 4️⃣ 30-day rainfall total
    # ========================
    last_30_days = daily_rain["date"].max() - pd.Timedelta(days=29)
    rain_30 = daily_rain[daily_rain["date"] >= last_30_days]["value"].sum()

    # ========================
    # 5️⃣ Heavy rain days (>25 mm)
    # ========================
    storm_days = daily_rain[daily_rain["value"] > 25].shape[0]

    # ========================
    # 6️⃣ 60-day rainfall anomaly
    # ========================
    recent_60_values = daily_rain[daily_rain["date"] >= daily_rain["date"].max() - pd.Timedelta(days=59)]["value"]
    recent_60 = recent_60_values.mean() if not recent_60_values.empty else 0

    previous_60_values = daily_rain[(daily_rain["date"] < daily_rain["date"].max() - pd.Timedelta(days=59)) &
                                    (daily_rain["date"] >= daily_rain["date"].max() - pd.Timedelta(days=119))]["value"]
    previous_60 = previous_60_values.mean() if not previous_60_values.empty else 0

    anomaly = recent_60 - previous_60

    # ========================
    # 7️⃣ Seasonal growth factor
    # ========================
    month = datetime.datetime.now().month
    if month in [3, 4, 5]:       # spring
        seasonal = 1.2
    elif month in [6, 7, 8]:     # summer
        seasonal = 1.1
    elif month in [9, 10, 11]:   # fall
        seasonal = 1.0
    else:                        # winter
        seasonal = 0.7

    # ========================
    # 8️⃣ Normalize & compute Weather Stress Score safely
    # ========================
    rain_values = daily_rain["value"]
    rain_norm = rain_30 / rain_values.sum() if rain_values.sum() > 0 else 0
    storm_norm = storm_days / len(rain_values) if len(rain_values) > 0 else 0

    if rain_values.max() != rain_values.min():
        anomaly_norm = (anomaly - rain_values.min()) / (rain_values.max() - rain_values.min())
    else:
        anomaly_norm = 0

    weather_stress = 0.5 * rain_norm + 0.3 * storm_norm + 0.2 * anomaly_norm

    # ========================
    # 9️⃣ Output results
    # ========================
    print("30-Day Rainfall Total (mm):", round(rain_30, 2))
    print("Heavy Rain Days (>25mm):", storm_days)
    print("60-Day Rainfall Anomaly (mm):", round(anomaly, 2))
    print("Seasonal Growth Factor:", seasonal)
    print("Weather Stress Score (0–1):", round(weather_stress, 2))

30-Day Rainfall Total (mm): 247.7
Heavy Rain Days (>25mm): 5
60-Day Rainfall Anomaly (mm): 9.57
Seasonal Growth Factor: 0.7
Weather Stress Score (0–1): 0.36


In [3]:
import requests
import pandas as pd
import numpy as np
import datetime

# ========================
# 1️⃣ NOAA API setup
# ========================
TOKEN = "YOUR_TOKEN_HERE"  # Replace with your NOAA token
headers = {"token": TOKEN}

# Set date range: past 1 year
end_date = datetime.datetime.now().date()
start_date = end_date - pd.Timedelta(days=365)

params = {
    "datasetid": "GHCND",
    "locationid": "FIPS:45079",  # Columbia, SC (Richland County)
    "startdate": start_date.isoformat(),
    "enddate": end_date.isoformat(),
    "datatypeid": "PRCP",
    "limit": 1000,
    "units": "metric"
}

# ========================
# 2️⃣ Pull data from NOAA
# ========================
response = requests.get(
    "https://www.ncei.noaa.gov/cdo-web/api/v2/data",
    headers=headers,
    params=params,
    timeout=10
)

data = response.json().get("results", [])
df = pd.DataFrame(data)

if df.empty:
    print("No data returned for this date range")
else:
    # convert to datetime and mm
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = df["value"] / 10  # tenths of mm → mm

    # ========================
    # 3️⃣ Aggregate stations by day
    # ========================
    daily_rain = df.groupby("date")["value"].sum().reset_index()

    # ========================
    # 4️⃣ 30-day rainfall total
    # ========================
    last_30_days = daily_rain["date"].max() - pd.Timedelta(days=29)
    rain_30 = daily_rain[daily_rain["date"] >= last_30_days]["value"].sum()

    # ========================
    # 5️⃣ Heavy rain days (>25 mm)
    # ========================
    storm_days = daily_rain[daily_rain["value"] > 25].shape[0]

    # ========================
    # 6️⃣ 60-day rainfall anomaly
    # ========================
    recent_60_values = daily_rain[daily_rain["date"] >= daily_rain["date"].max() - pd.Timedelta(days=59)]["value"]
    recent_60 = recent_60_values.mean() if not recent_60_values.empty else 0

    previous_60_values = daily_rain[(daily_rain["date"] < daily_rain["date"].max() - pd.Timedelta(days=59)) &
                                    (daily_rain["date"] >= daily_rain["date"].max() - pd.Timedelta(days=119))]["value"]
    previous_60 = previous_60_values.mean() if not previous_60_values.empty else 0

    anomaly = recent_60 - previous_60

    # ========================
    # 7️⃣ Seasonal growth factor
    # ========================
    month = datetime.datetime.now().month
    if month in [3, 4, 5]:       # spring
        seasonal = 1.2
    elif month in [6, 7, 8]:     # summer
        seasonal = 1.1
    elif month in [9, 10, 11]:   # fall
        seasonal = 1.0
    else:                        # winter
        seasonal = 0.7

    # ========================
    # 8️⃣ Normalize & compute Weather Stress Score safely
    # ========================
    rain_values = daily_rain["value"]
    rain_norm = rain_30 / rain_values.sum() if rain_values.sum() > 0 else 0
    storm_norm = storm_days / len(rain_values) if len(rain_values) > 0 else 0

    if rain_values.max() != rain_values.min():
        anomaly_norm = (anomaly - rain_values.min()) / (rain_values.max() - rain_values.min())
    else:
        anomaly_norm = 0

    weather_stress = 0.5 * rain_norm + 0.3 * storm_norm + 0.2 * anomaly_norm

    # ========================
    # 9️⃣ Output results
    # ========================
    print("30-Day Rainfall Total (mm):", round(rain_30, 2))
    print("Heavy Rain Days (>25mm):", storm_days)
    print("60-Day Rainfall Anomaly (mm):", round(anomaly, 2))
    print("Seasonal Growth Factor:", seasonal)
    print("Weather Stress Score (0–1):", round(weather_stress, 2))

No data returned for this date range


In [4]:
import requests
import pandas as pd
import numpy as np
import datetime

# ========================
# 1️⃣ NOAA API setup
# ========================
TOKEN = "qDMWPIYGHTKRKkhCWbKjwkUQxNMinWWd"  # Replace with your NOAA token
headers = {"token": TOKEN}

# Set date range: past 1 year
end_date = datetime.datetime.now().date()
start_date = end_date - pd.Timedelta(days=365)

params = {
    "datasetid": "GHCND",
    "locationid": "FIPS:45079",  # Columbia, SC (Richland County)
    "startdate": start_date.isoformat(),
    "enddate": end_date.isoformat(),
    "datatypeid": "PRCP",
    "limit": 1000,
    "units": "metric"
}

# ========================
# 2️⃣ Pull data from NOAA
# ========================
response = requests.get(
    "https://www.ncei.noaa.gov/cdo-web/api/v2/data",
    headers=headers,
    params=params,
    timeout=10
)

data = response.json().get("results", [])
df = pd.DataFrame(data)

if df.empty:
    print("No data returned for this date range")
else:
    # convert to datetime and mm
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = df["value"] / 10  # tenths of mm → mm

    # ========================
    # 3️⃣ Aggregate stations by day
    # ========================
    daily_rain = df.groupby("date")["value"].sum().reset_index()

    # ========================
    # 4️⃣ 30-day rainfall total
    # ========================
    last_30_days = daily_rain["date"].max() - pd.Timedelta(days=29)
    rain_30 = daily_rain[daily_rain["date"] >= last_30_days]["value"].sum()

    # ========================
    # 5️⃣ Heavy rain days (>25 mm)
    # ========================
    storm_days = daily_rain[daily_rain["value"] > 25].shape[0]

    # ========================
    # 6️⃣ 60-day rainfall anomaly
    # ========================
    recent_60_values = daily_rain[daily_rain["date"] >= daily_rain["date"].max() - pd.Timedelta(days=59)]["value"]
    recent_60 = recent_60_values.mean() if not recent_60_values.empty else 0

    previous_60_values = daily_rain[(daily_rain["date"] < daily_rain["date"].max() - pd.Timedelta(days=59)) &
                                    (daily_rain["date"] >= daily_rain["date"].max() - pd.Timedelta(days=119))]["value"]
    previous_60 = previous_60_values.mean() if not previous_60_values.empty else 0

    anomaly = recent_60 - previous_60

    # ========================
    # 7️⃣ Seasonal growth factor
    # ========================
    month = datetime.datetime.now().month
    if month in [3, 4, 5]:       # spring
        seasonal = 1.2
    elif month in [6, 7, 8]:     # summer
        seasonal = 1.1
    elif month in [9, 10, 11]:   # fall
        seasonal = 1.0
    else:                        # winter
        seasonal = 0.7

    # ========================
    # 8️⃣ Normalize & compute Weather Stress Score safely
    # ========================
    rain_values = daily_rain["value"]
    rain_norm = rain_30 / rain_values.sum() if rain_values.sum() > 0 else 0
    storm_norm = storm_days / len(rain_values) if len(rain_values) > 0 else 0

    if rain_values.max() != rain_values.min():
        anomaly_norm = (anomaly - rain_values.min()) / (rain_values.max() - rain_values.min())
    else:
        anomaly_norm = 0

    weather_stress = 0.5 * rain_norm + 0.3 * storm_norm + 0.2 * anomaly_norm

    # ========================
    # 9️⃣ Output results
    # ========================
    print("30-Day Rainfall Total (mm):", round(rain_30, 2))
    print("Heavy Rain Days (>25mm):", storm_days)
    print("60-Day Rainfall Anomaly (mm):", round(anomaly, 2))
    print("Seasonal Growth Factor:", seasonal)
    print("Weather Stress Score (0–1):", round(weather_stress, 2))

30-Day Rainfall Total (mm): 247.7
Heavy Rain Days (>25mm): 5
60-Day Rainfall Anomaly (mm): 9.57
Seasonal Growth Factor: 0.7
Weather Stress Score (0–1): 0.36
